# 201 · Group Sequential Design & SSR

Traditional A/B testing requires you to wait until a pre-determined sample size ($N_{fix}$) is reached. **Group Sequential Design (GSD)** allows you to perform "interim looks" and stop early for either success (**Efficacy**) or failure (**Futility**), while strictly controlling the Type I error rate.

In this tutorial, we will:
- Design a Binomial A/B test using the **Lan-DeMets (Spending Function)** approach.
- Perform sequential monitoring.
- **Advanced**: Use **Sample Size Re-estimation (SSR)** to adjust the design if the interim result is in the "Promising Zone".

## 1. Setup & Design

We use the `SpendingGST_LanDeMets1983` template. This template encapsulates the complex math of alpha-spending boundaries.

In [ ]:
import ibis
import numpy as np
import matplotlib.pyplot as plt
from earlysign.core.ledger import Ledger
from earlysign.schema.ES3.Binomial import ArmData
from earlysign.v1.templates.SpendingGST_LanDeMets1983 import LanDeMets1983Template

ledger = Ledger(ibis.connect("duckdb://:memory:"), "gsd_events").bind(exp_id="gsd_tutorial")
ledger.ensure()

# 1. Design the protocol
# 5 Looks, O-Brien Fleming boundaries.
trial = LanDeMets1983Template(ledger)
protocol = trial.design_binomial(
    p_control=0.10, 
    p_treatment=0.12,
    alpha=0.05, 
    power=0.8, 
    looks=5,
    spending_function="obrien_fleming"
)
trial.set_protocol(protocol)

print(f"Planned Max N: {int(protocol.method.stopping_policy.timer.max_sample_size)}")

## 2. Sequential Monitoring

We simulate data arriving in batches. After each `update()`, we check the progress.

In [ ]:
p_c, p_t = 0.10, 0.14  # True effect is larger than designed delta (0.02)
n_per_batch = 500

for look in range(1, 6):
    # 1. New data arrives
    batch = [
        ArmData(n=n_per_batch, success=np.random.binomial(n_per_batch, p_c), arm="C"),
        ArmData(n=n_per_batch, success=np.random.binomial(n_per_batch, p_t), arm="T")
    ]
    
    # 2. Analyze
    trial.update(batch)
    
    # 3. Decision Check
    report = trial.report_progress()
    print(f"Look {look}: N={report['sample_n']}, Z={report['z_stat']:.3f}, Status={report['status']}")
    
    if "STOP" in report['status']:
        print(f">>> STOPPED AT LOOK {look}!")
        break

## 3. Visualize the Trajectory

Templates provide built-in plotting to visualize the boundaries and the test statistic's path.

In [ ]:
trial.plot_result()
plt.show()

## 4. Advanced: Adaptive SSR (Promising Zone)

What if the interim result is not significant, but "looks promising"? Standard GSD would force you to continue with the original N, potentially under-powering the study if the true effect is smaller than expected.

**Sample Size Re-estimation (SSR)** allows you to increase $N$ at an interim look while preserving the Type I error using the **Cui-Hung-Wang (CHW)** weighted statistic.

In [ ]:
from earlysign.v1.templates.PromisingZone_CuiHungWang1999 import CuiHungWang1999Template

ssr_ledger = Ledger(ibis.connect("duckdb://:memory:"), "ssr_events").bind(exp_id="ssr_demo")
ssr_ledger.ensure()

ssr_trial = CuiHungWang1999Template(ssr_ledger)
ssr_protocol = ssr_trial.design_binomial(p_control=0.10, p_treatment=0.11, looks=2)
ssr_trial.set_protocol(ssr_protocol)

print(f"Original Max N: {int(ssr_protocol.method.stopping_policy.timer.max_sample_size)}")

# 1. Update with 'Promising' but not 'Significant' data
# CP (Conditional Power) will be low enough to justify SSR
mid_batch = [
    ArmData(n=1000, success=100, arm="C"),
    ArmData(n=1000, success=108, arm="T")
]
ssr_trial.update(mid_batch)

final_report = ssr_trial.report_progress()
print(f"Status: {final_report['status']}")
print(f"New Recommended Max N: {final_report['max_sample_size']}")

## 5. Summary

- **GSD**: Enables early stopping for efficient resource use.
- **Spending Functions**: Map boundaries to alpha expenditure over time.
- **Adaptive SSR**: Provides a "safety net" to recover power by increasing N in the Promising Zone.

In the next tutorial, we will explore **Anytime Valid Inference**, where we don't even need to pre-specify the number of looks.